# Customer Churn Prediction — Exploratory Data Analysis

Explore the IBM Telco Customer Churn dataset before model development. This notebook is designed to run in Google Colab and can also run from a local repository checkout.

## 1. Runtime and dataset setup

The repository is cloned into the Colab runtime when it is not already the current working directory. The raw dataset is intentionally not committed; upload the original CSV when prompted.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

current_dir = Path.cwd()
PROJECT_ROOT = next(
    (candidate for candidate in (current_dir, *current_dir.parents)
     if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    colab_root = Path("/content")
    if not colab_root.exists():
        raise RuntimeError(
            "Run this notebook from the project root or its notebooks/ directory. "
            "In Google Colab, open it from the GitHub Colab link."
        )
    PROJECT_ROOT = colab_root / "customer-churn-prediction"
    if not (PROJECT_ROOT / "src").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Expected dataset: {DATA_PATH}")

In [ ]:
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Dataset not found at {DATA_PATH}. Download the Kaggle CSV and place it there."
        ) from exc

    print(f"Upload the original file named {EXPECTED_CSV}")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if EXPECTED_CSV in uploaded:
        selected_name = EXPECTED_CSV
    elif len(csv_names) == 1:
        selected_name = csv_names[0]
    else:
        raise ValueError(f"Upload exactly one CSV, preferably named {EXPECTED_CSV}. Found: {csv_names}")
    if selected_name != EXPECTED_CSV:
        (DATA_PATH.parent / EXPECTED_CSV).write_bytes(uploaded[selected_name])
    else:
        DATA_PATH.write_bytes(uploaded[selected_name])

assert DATA_PATH.exists(), "Dataset setup did not create the expected CSV file."

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
raw_df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
display(raw_df.head())
raw_df.info()

## 2. Data quality checks

Keep the raw dataframe unchanged. Convert `TotalCharges` only in the analysis copy. Blank charges for zero-tenure customers represent customers with no billed tenure; other missing values remain missing for the later training pipeline to handle.

In [ ]:
df = raw_df.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
zero_tenure_missing = df["TotalCharges"].isna() & df["tenure"].eq(0)
df.loc[zero_tenure_missing, "TotalCharges"] = 0.0

print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
print("Duplicate rows:", int(df.duplicated().sum()))
print("TotalCharges blanks converted to zero for zero-tenure customers:", int(zero_tenure_missing.sum()))
print("Other TotalCharges values still missing:", int(df["TotalCharges"].isna().sum()))

## 3. Target distribution

Churn is the binary prediction target. Record the observed class counts and proportions rather than assuming a fixed class balance.

In [ ]:
target_counts = df["Churn"].value_counts(dropna=False)
target_share = df["Churn"].value_counts(normalize=True, dropna=False).mul(100)
display(pd.DataFrame({"customers": target_counts, "percentage": target_share.round(2)}))
churn_rate = df["Churn"].eq("Yes").mean() * 100
print(f"Observed churn rate: {churn_rate:.2f}%")

ax = sns.countplot(data=df, x="Churn", order=["No", "Yes"])
ax.set(title="Customer churn distribution", xlabel="Churn", ylabel="Customers")
plt.tight_layout()
plt.show()

## 4. Numeric feature distributions

Compare tenure and billing distributions between customers who stayed and customers who churned.

In [ ]:
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
display(df[numeric_columns].describe().T)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for column, axis in zip(numeric_columns, axes):
    sns.histplot(data=df, x=column, hue="Churn", kde=True, element="step", ax=axis)
    axis.set_title(f"{column} by churn")
plt.tight_layout()
plt.show()

## 5. Churn rates across customer and service categories

Rates are calculated within each category. A high rate is an association in this dataset, not proof that the category causes churn.

In [ ]:
categorical_columns = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod"
]
churn_rate_tables = {}
for column in categorical_columns:
    rates = df.groupby(column, dropna=False)["Churn"].apply(lambda values: values.eq("Yes").mean()).sort_values(ascending=False)
    churn_rate_tables[column] = rates
    display((rates.mul(100).round(2)).rename("churn_rate_percent").to_frame())
    axis = sns.barplot(x=rates.index.astype(str), y=rates.values * 100)
    axis.set(title=f"Churn rate by {column}", xlabel=column, ylabel="Churn rate (%)")
    axis.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

## 6. Numeric correlations

Correlation describes linear association among numeric variables. It is not a measure of causal influence.

In [ ]:
numeric_for_corr = df[numeric_columns].copy()
numeric_for_corr["ChurnBinary"] = df["Churn"].map({"No": 0, "Yes": 1})
correlation = numeric_for_corr.corr(numeric_only=True)
display(correlation.round(3))
plt.figure(figsize=(7, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Numeric feature and target correlations")
plt.tight_layout()
plt.show()

## 7. Initial findings and modeling considerations

The following cell prints observed category churn rates for selected features. Use the output to write conclusions after running the notebook; do not treat these associations as causal explanations.

In [ ]:
for column in ["Contract", "PaymentMethod", "InternetService", "TechSupport", "OnlineSecurity"]:
    rates = churn_rate_tables[column].mul(100).round(2)
    print(f"{column}: highest observed category churn rate = {rates.index[0]} ({rates.iloc[0]:.2f}%)")

print("\nModeling notes:")
print("- Exclude customerID because it is an identifier.")
print("- Convert TotalCharges to numeric and handle remaining missing values inside the training pipeline.")
print("- Split the data before fitting imputers, encoders, or scalers.")
print("- Use cross-validation on the training split for model selection.")
print("- Reserve the held-out test split for final evaluation.")

## 8. Conclusion

After running all cells, summarize the strongest observed patterns here. Distinguish descriptive associations from model-based feature importance, and do not claim that any variable causes churn based only on this EDA.

In [ ]:
print(f"Dataset rows: {len(df):,}")
print(f"Features excluding target and identifier: {df.shape[1] - 2}")
print(f"Observed churn rate: {churn_rate:.2f}%")
print(f"Duplicate rows: {int(df.duplicated().sum())}")
print("EDA complete. Record interpretation based on the displayed tables and charts.")